# Preprocessing

Loads the UN General Debate speeches, adds speaker and country metadata and makes a lightly cleaned `SpeechClean`
column. Notebook 02 uses it for keyword matching, sentence splitting and spaCy.

In [1]:
import os
import re

import pandas as pd

## Load speeches

The country code, session and year come from each file name. Files are read with `utf-8-sig`, so the byte-order mark
at the start of some files is removed.

In [2]:
txt_folder = '../datasets/TXT'

rows = []
for session_folder in sorted(os.listdir(txt_folder)):
    folder_path = os.path.join(txt_folder, session_folder)
    for filename in sorted(os.listdir(folder_path)):
        iso_code, session, year = filename.replace('.txt', '').split('_')
        file_path = os.path.join(folder_path, filename)
        with open(file_path, encoding='utf-8-sig') as f:
            speech = f.read()
        rows.append({
            'Session': int(session),
            'Year': int(year),
            'ISO-alpha3 Code': iso_code,
            'Speech': speech
        })

df = pd.DataFrame(rows)
n_speeches_loaded = len(df)
print(f"Loaded {n_speeches_loaded} speeches")
df.head()

Loaded 11141 speeches


,Session,Year,ISO-alpha3 Code,Speech
0,1,1946,ARG,At the resumption of the first session of the ...
1,1,1946,AUS,The General Assembly of the United Nations is ...
2,1,1946,BEL,The\tprincipal organs of the United Nations ha...
3,1,1946,BLR,As more than a year has elapsed since the Unit...
4,1,1946,BOL,Coming to this platform where so many distingu...


## Add speaker metadata

A few speaker rows share the same (country, session, year) key and would duplicate speeches in the merge, so we print
and drop them. The left join keeps every speech.

In [3]:
speakers = pd.read_excel('../datasets/Speakers_by_session.xlsx')
speakers = speakers.rename(columns={'ISO Code': 'ISO-alpha3 Code', 'Name of Person Speaking': 'SpeakerName'})
speakers = speakers[['Session', 'Year', 'ISO-alpha3 Code', 'SpeakerName', 'Post']]

key_cols = ['ISO-alpha3 Code', 'Session', 'Year']
dup_keys = speakers[speakers.duplicated(subset=key_cols, keep=False)].sort_values(key_cols)
print(f"{dup_keys[key_cols].drop_duplicates().shape[0]} duplicated speaker keys:")
print(dup_keys)

speakers = speakers.drop_duplicates(subset=key_cols)

df = df.merge(speakers, on=key_cols, how='left')
print(df.shape)
df.head()

17 duplicated speaker keys:
       Session  Year ISO-alpha3 Code               SpeakerName  \
10542       13  1958             BGR               Mr. Lukanov   
10564       13  1958             BGR               Mr. Lukanov   
9451        24  1969             CMR                 Mr. NJINE   
9539        24  1969             CMR        Mr. AHMADCU AHIDJO   
10588       12  1957             CSK                Mr. DAVID    
10626       12  1957             CSK                 Mr. DAVID   
10510       13  1958             CSK                 Mr. David   
10566       13  1958             CSK                 Mr. David   
10354       15  1960             DNK                  Mr. KRAG   
10414       15  1960             DNK    H.M. King FREDERIK IX    
10333       15  1960             ETH          HAILE SELASSIE X   
10367       15  1960             ETH             Mr. ABTE WOLD   
10174       17  1962             GIN               Mr. LANSANA   
10234       17  1962             GIN           M

,Session,Year,ISO-alpha3 Code,Speech,SpeakerName,Post
0,1,1946,ARG,At the resumption of the first session of the ...,Mr. Arce,NaN
1,1,1946,AUS,The General Assembly of the United Nations is ...,Mr. Makin,NaN
2,1,1946,BEL,The\tprincipal organs of the United Nations ha...,Mr. Van Langenhove,NaN
3,1,1946,BLR,As more than a year has elapsed since the Unit...,Mr. Kiselev,NaN
4,1,1946,BOL,Coming to this platform where so many distingu...,Mr. Costa du Rels,NaN


## Add UNSD M49 regions

Region and sub-region names are added with a left join, so no speech is lost. We print the countries without a region:
former states such as Yugoslavia, and the EU.

In [4]:
unsd = pd.read_csv('../datasets/UNSD — Methodology.csv', sep=';')
unsd = unsd[['ISO-alpha3 Code', 'Country or Area', 'Region Name', 'Sub-region Name', 'Least Developed Countries (LDC)']]

df = df.merge(unsd, on='ISO-alpha3 Code', how='left')

missing_region = df[df['Region Name'].isna()]
print(f"{len(missing_region)} speeches have no Region Name")
print(
    missing_region.groupby('ISO-alpha3 Code')['Year']
    .agg(n_speeches='count', first_year='min', last_year='max')
)

print(df.shape)
df.head()

150 speeches have no Region Name
                 n_speeches  first_year  last_year
ISO-alpha3 Code                                   
CSK                      46        1946       1992
DDR                      18        1973       1990
EU                       14        2011       2024
YMD                      21        1968       1989
YUG                      51        1946       2005
(11141, 10)


,Session,Year,ISO-alpha3 Code,Speech,SpeakerName,Post,Country or Area,Region Name,Sub-region Name,Least Developed Countries (LDC)
0,1,1946,ARG,At the resumption of the first session of the ...,Mr. Arce,NaN,Argentina,Americas,Latin America and the Caribbean,NaN
1,1,1946,AUS,The General Assembly of the United Nations is ...,Mr. Makin,NaN,Australia,Oceania,Australia and New Zealand,NaN
2,1,1946,BEL,The\tprincipal organs of the United Nations ha...,Mr. Van Langenhove,NaN,Belgium,Europe,Western Europe,NaN
3,1,1946,BLR,As more than a year has elapsed since the Unit...,Mr. Kiselev,NaN,Belarus,Europe,Eastern Europe,NaN
4,1,1946,BOL,Coming to this platform where so many distingu...,Mr. Costa du Rels,NaN,Bolivia (Plurinational State of),Americas,Latin America and the Caribbean,NaN


## Light text cleaning (`SpeechClean`)

Some words are split over two lines with a hyphen (e.g. "renew-\nable"). We join them again, but only with letters on
both sides, so a page number after a hyphen is not glued to a word. Then tabs, newlines and double spaces become single
spaces.

Case and punctuation are kept: sentence splitting needs capitals and full stops, and the keyword matching ignores case
anyway. The raw `Speech` column stays unchanged.

In [5]:
HYPHEN_LINEBREAK = r'([^\W\d_]+)-\n([^\W\d_]+)'

hyphen_hits = [m.group(0) for text in df['Speech'] for m in re.finditer(HYPHEN_LINEBREAK, text)]
print(f"{len(hyphen_hits)} hyphenated line-break words found")
print(hyphen_hits[:3])

df['SpeechClean'] = df['Speech'].str.replace(HYPHEN_LINEBREAK, r'\1\2', regex=True)
df['SpeechClean'] = df['SpeechClean'].str.replace(r'\s+', ' ', regex=True).str.strip()

df[['Speech', 'SpeechClean']].head()

7891 hyphenated line-break words found
['re-\nflexion', 'in-\nvestment', 'Viet-\nNamese']


,Speech,SpeechClean
0,At the resumption of the first session of the ...,At the resumption of the first session of the ...
1,The General Assembly of the United Nations is ...,The General Assembly of the United Nations is ...
2,The\tprincipal organs of the United Nations ha...,The principal organs of the United Nations hav...
3,As more than a year has elapsed since the Unit...,As more than a year has elapsed since the Unit...
4,Coming to this platform where so many distingu...,Coming to this platform where so many distingu...


## Word count

`WordCount` is used later to turn keyword counts into rates, because a long speech contains more matches than a short
one.

In [6]:
df['WordCount'] = df['SpeechClean'].str.split().str.len()
df['WordCount'].describe()

count    11141.000000
mean      2900.514406
std       1496.752240
min        423.000000
25%       1864.000000
50%       2550.000000
75%       3622.000000
max      22003.000000
Name: WordCount, dtype: float64

## No stop-word removal

We do not remove stop words. Stop-word lists remove words like "will", "may" and "could", which are exactly the words
our caution measure looks for.

## Checks

No speech is lost or duplicated in the merges, every (year, country) pair is unique, and no speech is empty after
cleaning.

In [7]:
assert len(df) == n_speeches_loaded, "Row count changed after merges"
assert not df.duplicated(subset=['Year', 'ISO-alpha3 Code']).any(), "Duplicate (Year, ISO-alpha3 Code) keys"
assert (df['SpeechClean'].str.len() > 0).all(), "Empty SpeechClean found"

print(f"Year range: {df['Year'].min()}-{df['Year'].max()}")
print(f"Unique countries: {df['ISO-alpha3 Code'].nunique()}")

Year range: 1946-2025
Unique countries: 200


## Save

The index is (Year, ISO-alpha3 Code), and the table is saved as Parquet for notebook 02.

In [8]:
df = df.set_index(['Year', 'ISO-alpha3 Code'])
df.to_parquet('../datasets/speeches_simple_clean.parquet')
print(df.shape)

(11141, 10)


## Final columns

- Year (index)
- ISO-alpha3 Code (index)
- Session
- Speech
- SpeakerName
- Post
- Country or Area
- Region Name
- Sub-region Name
- Least Developed Countries (LDC)
- SpeechClean
- WordCount